# WFP country food prices → landing_dev and raw_dev

Reads `wfp_food_prices_<iso3>.csv` from `data-eng/ingestion/data_csv`. Does not call Humdata.

- Guide: [`wfp_hbx_46_data_upload.md`](./wfp_hbx_46_data_upload.md)
- One file is one table: `hdx_wfp_food_prices_<iso3>`
- `landing_dev` keeps the publisher columns. `raw_dev` keeps the typed transform plus country.

## Setup

In [ ]:
import sys
from pathlib import Path


def _load_notebook_bootstrap() -> None:
    here = Path.cwd().resolve()
    candidates: list[Path] = []
    for base in [here, *here.parents]:
        candidates.extend(
            [
                base / "_bootstrap.py",
                base / "notebooks" / "_bootstrap.py",
                base / "ingestion" / "notebooks" / "_bootstrap.py",
                base / "data-eng" / "ingestion" / "notebooks" / "_bootstrap.py",
            ]
        )
    for boot in candidates:
        if boot.is_file():
            nb_dir = str(boot.parent)
            if nb_dir not in sys.path:
                sys.path.insert(0, nb_dir)
            from _bootstrap import ensure_ingestion

            print("ingestion ready", ensure_ingestion())
            return
    raise ModuleNotFoundError("Could not find notebooks/_bootstrap.py")


_load_notebook_bootstrap()

from ingestion.common.auth import ensure_auth_safe
from ingestion.common.bq import get_client
from ingestion.common.config import get_settings
from ingestion.common.load import add_metadata, load_dataframe, new_run_id

KEY_COLS = ["date", "market_id", "commodity_id", "pricetype", "unit"]
COUNTRY_NAMES = {
    "AGO": "Angola",
    "BDI": "Burundi",
    "BEN": "Benin",
    "BFA": "Burkina Faso",
    "CAF": "Central African Republic",
    "CIV": "Côte d'Ivoire",
    "CMR": "Cameroon",
    "COD": "Democratic Republic of the Congo",
    "COG": "Congo",
    "CPV": "Cabo Verde",
    "DJI": "Djibouti",
    "DZA": "Algeria",
    "EGY": "Egypt",
    "ERI": "Eritrea",
    "ETH": "Ethiopia",
    "GAB": "Gabon",
    "GHA": "Ghana",
    "GIN": "Guinea",
    "GMB": "Gambia",
    "GNB": "Guinea-Bissau",
    "KEN": "Kenya",
    "LBR": "Liberia",
    "LBY": "Libya",
    "LSO": "Lesotho",
    "MDG": "Madagascar",
    "MLI": "Mali",
    "MOZ": "Mozambique",
    "MRT": "Mauritania",
    "MWI": "Malawi",
    "NAM": "Namibia",
    "NER": "Niger",
    "NGA": "Nigeria",
    "RWA": "Rwanda",
    "SDN": "Sudan",
    "SEN": "Senegal",
    "SLE": "Sierra Leone",
    "SOM": "Somalia",
    "SSD": "South Sudan",
    "SWZ": "Eswatini",
    "TCD": "Chad",
    "TGO": "Togo",
    "TZA": "Tanzania",
    "UGA": "Uganda",
    "ZAF": "South Africa",
    "ZMB": "Zambia",
    "ZWE": "Zimbabwe",
}

landing_settings = get_settings(cli_target="landing_dev")
raw_settings = get_settings(cli_target="raw_dev")
ensure_auth_safe(landing_settings)
run_id = new_run_id()
print("python", sys.executable)
print("project", landing_settings.project, "run_id", run_id)
print("landing", landing_settings.target_dataset, "raw", raw_settings.target_dataset)

## Load each local CSV

Skip a file whose first line is not the publisher header. A missing folder or no matching CSV loads nothing.

In [ ]:
from io import StringIO

import pandas as pd

from ingestion.sources.hdx_wfp_prices.transform import _skip_hxl, transform


def _data_dir() -> Path | None:
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for candidate in (
            base / "data_csv",
            base / "ingestion" / "data_csv",
            base / "data-eng" / "ingestion" / "data_csv",
        ):
            if candidate.is_dir():
                return candidate
    return None


def _iso3(path: Path) -> str | None:
    stem = path.stem
    prefix = "wfp_food_prices_"
    if not stem.startswith(prefix):
        return None
    code = stem[len(prefix):].upper()
    if code not in COUNTRY_NAMES:
        return None
    return code


data_dir = _data_dir()
files = sorted(data_dir.glob("wfp_food_prices_*.csv")) if data_dir else []
if data_dir is None or not files:
    print("no CSVs to load", data_dir)
else:
    client = get_client(landing_settings)
    loaded: list[str] = []
    skipped: list[str] = []
    for path in files:
        iso3 = _iso3(path)
        text = path.read_text(encoding="utf-8-sig", errors="replace")
        first = text.splitlines()[0] if text.strip() else ""
        if iso3 is None or not first.startswith("date,"):
            skipped.append(path.name)
            print("skip", path.name)
            continue
        landing_df = pd.read_csv(StringIO(_skip_hxl(text)))
        if landing_df.empty:
            skipped.append(path.name)
            print("skip empty", path.name)
            continue
        raw_df = transform(None, landing_df)
        raw_df["country_iso3"] = iso3
        raw_df["country_name"] = COUNTRY_NAMES[iso3]
        table = f"hdx_wfp_food_prices_{iso3.lower()}"
        source_url = str(path.resolve())
        landing_ready = add_metadata(
            landing_df,
            source=table,
            run_id=run_id,
            source_url=source_url,
            key_cols=KEY_COLS,
        )
        raw_ready = add_metadata(
            raw_df,
            source=table,
            run_id=run_id,
            source_url=source_url,
            key_cols=KEY_COLS,
        )
        landing_result = load_dataframe(
            client,
            landing_settings,
            landing_ready,
            table=table,
            mode="replace_partition",
            run_id=run_id,
        )
        raw_result = load_dataframe(
            client,
            raw_settings,
            raw_ready,
            table=table,
            mode="replace_partition",
            run_id=run_id,
        )
        loaded.append(table)
        print(table, "landing", landing_result.rows, "raw", raw_result.rows)
    print("loaded", len(loaded), "skipped", skipped)